In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
import numpy as np

In [2]:
def process_house_data(file_path):
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import MinMaxScaler


    df = pd.read_csv(file_path)
    df['Time'] = pd.to_datetime(df['Time'])
    df['Weekday'] = df['Time'].dt.day_name()


    max_index = df['Aggregate'].idxmax()
    df = df.drop(index=max_index)
    df = df.dropna()
    df['Time'] = pd.to_datetime(df['Time'])


    filtered_data = df[(df['Time'].dt.hour >= 0) & (df['Time'].dt.hour < 10)]
    ex1 = filtered_data[filtered_data['Aggregate'] <= 10000]
    ex1['Time'] = pd.to_datetime(ex1['Time'])
    ex1['Hour'] = ex1['Time'].dt.hour
    ex1['Date'] = ex1['Time'].dt.date


    ex2 = ex1.drop(['Date'], axis=1)
    ex2['Time'] = pd.to_datetime(ex2['Time'])
    ex2['Year'] = ex2['Time'].dt.year
    ex2['Month'] = ex2['Time'].dt.month
    ex2['Day'] = ex2['Time'].dt.day
    ex2['Hour'] = ex2['Time'].dt.hour


    def get_season(month):
        if month in [3, 4, 5]:
            return 'Spring'
        elif month in [6, 7, 8]:
            return 'Summer'
        elif month in [9, 10, 11]:
            return 'Autumn'
        else:
            return 'Winter'

    ex2['Season'] = ex2['Month'].apply(get_season)
    season_dummies = pd.get_dummies(ex2['Season'], prefix='Season')
    ex2 = pd.concat([ex2, season_dummies], axis=1)
    ex2.drop('Season', axis=1, inplace=True)


    hourly_variance = ex2.groupby(['Year', 'Month', 'Day', 'Hour'])['Aggregate'].var().reset_index()
    ex2 = ex2.merge(hourly_variance, on=['Year', 'Month', 'Day', 'Hour'], how='left', suffixes=('', '_Hourly_Variance'))

    for col in ['Season_Spring', 'Season_Summer', 'Season_Autumn', 'Season_Winter']:
        if col in ex2.columns:
            ex2[col] = ex2[col].astype(int)
        else:
            ex2[col] = 0


    ex3 = ex2.copy()
    ex3['Time'] = pd.to_datetime(ex3['Time'])
    if 'Unix' not in ex3.columns:
        ex3['Unix'] = ex3['Time'].astype(np.int64) // 10**9

    ex3['Load_Derivative'] = ex3['Aggregate'].diff() / ex3['Unix'].diff()
    ex3['Minute'] = ex3['Time'].dt.floor('30T')


    ex3['Max_Load_Derivative_60min'] = ex3['Load_Derivative'].rolling(window=60, min_periods=1).apply(lambda x: np.max(np.abs(x)))


    ex4 = ex3.groupby(['Minute']).agg({
        'Aggregate': ['sum', 'mean'],
        'Load_Derivative': 'mean',
        'Max_Load_Derivative_60min': 'mean',
        'Weekday': 'first',
        'Season_Spring': 'first',
        'Season_Summer': 'first',
        'Season_Autumn': 'first',
        'Season_Winter': 'first'
    }).reset_index()


    ex4.columns = ['Minute', 'Aggregate_sum', 'Aggregate_mean',
                   'Load_Derivative', 'Max_Load_Derivative_60min', 
                   'Weekday', 'Season_Spring', 'Season_Summer', 'Season_Autumn', 'Season_Winter']


    ex4['60min_variance'] = ex4['Aggregate_mean'].rolling(window=2, min_periods=1).var()

    def mean_rate_of_change(series):
        return np.mean(np.diff(series))

    ex4['MRoC_60min'] = ex4['Aggregate_mean'].rolling(window=2).apply(mean_rate_of_change, raw=True)


    ex4['Minute'] = pd.to_datetime(ex4['Minute'])
    ex4['hour'] = ex4['Minute'].dt.hour
    ex4['month'] = ex4['Minute'].dt.month

    weekday_dummies = pd.get_dummies(ex4['Weekday'], prefix='Weekday')
    ex4 = pd.concat([ex4.drop('Weekday', axis=1), weekday_dummies], axis=1)


    numerical_columns = ['Load_Derivative', '60min_variance', 'MRoC_60min', 
                         'Max_Load_Derivative_60min']
    ex4 = ex4.dropna(subset=numerical_columns)
    scaler = MinMaxScaler()
    ex4[numerical_columns] = scaler.fit_transform(ex4[numerical_columns])

  
    ex4['Date'] = pd.to_datetime(ex4['Minute'])
    ex4.set_index('Date', inplace=True)
    ex4['hour_sin'] = np.sin(2 * np.pi * ex4['hour'] / 24)
    ex4['hour_cos'] = np.cos(2 * np.pi * ex4['hour'] / 24)

    return ex4




In [5]:
ex4 = process_house_data('House_11.csv')
ex3 = process_house_data('House_18.csv')
ex2 = process_house_data('House_8.csv')
ex1 = process_house_data('House_6.csv')
ex0 = process_house_data('House_4.csv')

/var/folders/tb/y4xkhqb55clb52ckyg4fsdg40000gn/T/ipykernel_78636/3293906744.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ex1['Time'] = pd.to_datetime(ex1['Time'])
/var/folders/tb/y4xkhqb55clb52ckyg4fsdg40000gn/T/ipykernel_78636/3293906744.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ex1['Hour'] = ex1['Time'].dt.hour
/var/folders/tb/y4xkhqb55clb52ckyg4fsdg40000gn/T/ipykernel_78636/3293906744.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try

In [6]:
ex4

,Minute,Aggregate_sum,Aggregate_mean,Load_Derivative,Max_Load_Derivative_60min,Season_Spring,Season_Summer,Season_Autumn,Season_Winter,60min_variance,...,month,Weekday_Friday,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday,hour_sin,hour_cos
Date,,,,,,,,,,,,,,,,,,,,,
2014-06-04 00:30:00,2014-06-04 00:30:00,51337,171.695652,0.603993,0.003015,0,1,0,0,0.009069,...,6,False,False,False,False,False,False,True,0.000000,1.000000
2014-06-04 01:00:00,2014-06-04 01:00:00,61409,204.696667,0.601454,0.002183,0,1,0,0,0.000169,...,6,False,False,False,False,False,False,True,0.258819,0.965926
2014-06-04 01:30:00,2014-06-04 01:30:00,38462,128.206667,0.600626,0.002185,0,1,0,0,0.000909,...,6,False,False,False,False,False,False,True,0.258819,0.965926
2014-06-04 02:00:00,2014-06-04 02:00:00,21889,73.207358,0.595947,0.005650,0,1,0,0,0.000470,...,6,False,False,False,False,False,False,True,0.500000,0.866025
2014-06-04 02:30:00,2014-06-04 02:30:00,19759,67.207483,0.601826,0.004358,0,1,0,0,0.000006,...,6,False,False,False,False,False,False,True,0.500000,0.866025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-06-30 07:30:00,2015-06-30 07:30:00,58236,223.984615,0.618237,0.005116,0,1,0,0,0.000800,...,6,False,False,False,False,False,True,False,0.965926,-0.258819
2015-06-30 08:00:00,2015-06-30 08:00:00,404781,1679.589212,0.624925,0.012274,0,1,0,0,0.329202,...,6,False,False,False,False,False,True,False,0.866025,-0.500000
2015-06-30 08:30:00,2015-06-30 08:30:00,439083,1986.800905,0.609570,0.008785,0,1,0,0,0.014664,...,6,False,False,False,False,False,True,False,0.866025,-0.500000


In [ ]:
import tensorflow as tf
from tensorflow.keras.losses import MeanAbsoluteError



In [ ]:
df =ex4



In [7]:
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, BatchNormalization, LSTM, RepeatVector, Attention, Dense, TimeDistributed, Flatten
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

features = ['Aggregate_mean', 'Load_Derivative', 'Max_Load_Derivative_60min','60min_variance', 'MRoC_60min', 'hour', 'month','Season_Spring','Season_Summer','Season_Autumn','Season_Winter','Weekday_Friday','Weekday_Monday','Weekday_Saturday','Weekday_Sunday','Weekday_Thursday','Weekday_Tuesday','Weekday_Wednesday','hour_sin','hour_cos']


train_data = [ex0, ex1, ex2, ex4]
test_data = [ex3]

step_per_day = 20
input_days = 14
forecast_horizon = 20


train_concat = pd.concat(train_data, axis=0)
scaler = MinMaxScaler()
scaler.fit(train_concat[features])


X_house_list, y_house_list = [], []


for house_data in train_data:
    X_list, y_list = [], []
    anchors = house_data.index[house_data.index.time == pd.to_datetime('00:00').time()].tolist()
    for i in range(len(anchors) - input_days - 1):
        input_start = anchors[i]
        input_end = anchors[i + input_days]
        target_start = input_end
        target_end = anchors[i + input_days + 1]

        input_window = house_data.loc[input_start:input_end - pd.Timedelta(minutes=20)]
        target_window = house_data.loc[target_start:target_end - pd.Timedelta(minutes=20)]

        if len(input_window) == step_per_day * input_days and len(target_window) == forecast_horizon:
            input_scaled = scaler.transform(input_window[features])
            target_scaled = scaler.transform(target_window[features])
            X_list.append(input_scaled)
            y_list.append(target_scaled[:, features.index('Aggregate_mean')])
    
    X_house_list.append(np.array(X_list))
    y_house_list.append(np.array(y_list))


X_train = np.concatenate(X_house_list, axis=0)
y_train = np.concatenate(y_house_list, axis=0)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


X_test_list, y_test_list = [], []

for house_data in test_data:
    anchors = house_data.index[house_data.index.time == pd.to_datetime('00:00').time()].tolist()
    for i in range(len(anchors) - input_days - 1):
        input_start = anchors[i]
        input_end = anchors[i + input_days]
        target_start = input_end
        target_end = anchors[i + input_days + 1]

        input_window = house_data.loc[input_start:input_end - pd.Timedelta(minutes=20)]
        target_window = house_data.loc[target_start:target_end - pd.Timedelta(minutes=20)]

        if len(input_window) == step_per_day * input_days and len(target_window) == forecast_horizon:
            input_scaled = scaler.transform(input_window[features])
            target_scaled = scaler.transform(target_window[features])
            X_test_list.append(input_scaled)
            y_test_list.append(target_scaled[:, features.index('Aggregate_mean')])


X_test = np.array(X_test_list)
y_test = np.array(y_test_list)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


house_data = list(zip(X_house_list, y_house_list))
house_data_shuffled = shuffle(house_data, random_state=42)

X_train = np.concatenate([x for x, _ in house_data_shuffled], axis=0)
y_train = np.concatenate([y for _, y in house_data_shuffled], axis=0)


@tf.function
def soft_dtw_distance(y_true, y_pred, gamma=1.0):
    if len(y_true.shape) == 3:
        y_true = tf.squeeze(y_true, axis=-1)
    if len(y_pred.shape) == 3:
        y_pred = tf.squeeze(y_pred, axis=-1)
    
    batch_size = tf.shape(y_true)[0]
    seq_len = tf.shape(y_true)[1]

    def pairwise_dist(a, b):
        a = tf.expand_dims(a, 2)
        b = tf.expand_dims(b, 1)
        return tf.square(a - b)

    D = pairwise_dist(y_true, y_pred)

    R = tf.ones_like(D) * 1e8
    R = tf.tensor_scatter_nd_update(
        R,
        tf.concat([
            tf.expand_dims(tf.range(batch_size), axis=1),
            tf.zeros((batch_size, 1), dtype=tf.int32),
            tf.zeros((batch_size, 1), dtype=tf.int32)
        ], axis=1),
        tf.zeros((batch_size,))
    )

    for i in tf.range(1, seq_len):
        for j in tf.range(1, seq_len):
            r0 = R[:, i - 1, j - 1]
            r1 = R[:, i - 1, j]
            r2 = R[:, i, j - 1]
            r_min = tf.minimum(tf.minimum(r0, r1), r2)
            update = D[:, i, j] + r_min
            R = tf.tensor_scatter_nd_update(
                R,
                tf.concat([
                    tf.expand_dims(tf.range(batch_size), axis=1),
                    tf.fill((batch_size, 1), i),
                    tf.fill((batch_size, 1), j)
                ], axis=1),
                update
            )

    return tf.reduce_mean(R[:, -1, -1])


def combined_loss(alpha=0.5, beta=0.5):
    mae_fn = tf.keras.losses.MeanAbsoluteError()
    def loss_fn(y_true, y_pred):
        mae = mae_fn(y_true, y_pred)
        dtw = soft_dtw_distance(y_true, y_pred)
        return alpha * mae + beta * dtw
    return loss_fn

tscv = TimeSeriesSplit(n_splits=5)

from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Dense, Dropout, GlobalAveragePooling1D, Add

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    # Multi-head self-attention
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=head_size, dropout=dropout)(inputs, inputs)
    attention = Dropout(dropout)(attention)
    attention_out = Add()([inputs, attention])
    attention_out = LayerNormalization(epsilon=1e-6)(attention_out)

 
    ff = Dense(ff_dim, activation="relu")(attention_out)
    ff = Dropout(dropout)(ff)
    ff = Dense(inputs.shape[-1])(ff)
    ff_out = Add()([attention_out, ff])
    ff_out = LayerNormalization(epsilon=1e-6)(ff_out)
    
    return ff_out




for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    print(f"\nFold {fold+1}")
    
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

 

    input_seq = Input(shape=(X_train.shape[1], X_train.shape[2]))
    x = transformer_encoder(input_seq, head_size=64, num_heads=4, ff_dim=256, dropout=0.1)
    x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=256, dropout=0.1)
    x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=256, dropout=0.1)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.1)(x)
    output = Dense(forecast_horizon)(x)

    model = Model(inputs=input_seq, outputs=output)

    model.compile(optimizer='adam', loss=combined_loss(), metrics=['mae'])

  
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
        ModelCheckpoint(f'best_model_fold{fold+1}.keras', monitor='val_loss', save_best_only=True)
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=8,
        callbacks=callbacks,
        verbose=2
    )


y_pred = model.predict(X_test)


y_pred_inv = scaler.inverse_transform(
    np.concatenate([y_pred.reshape(-1, 1)] + [np.zeros((y_pred.size, len(features) - 1))], axis=1)
)[:, 0].reshape(y_pred.shape)

y_test_inv = scaler.inverse_transform(
    np.concatenate([y_test.reshape(-1, 1)] + [np.zeros((y_test.size, len(features) - 1))], axis=1)
)[:, 0].reshape(y_test.shape)

mse = mean_squared_error(y_test_inv.flatten(), y_pred_inv.flatten())
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_inv.flatten(), y_pred_inv.flatten())

print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MSE: {mse:.6f}")


plt.figure(figsize=(10, 4))
plt.plot(y_test_inv[0], label='truth')
plt.plot(y_pred_inv[0], label='predict')
plt.legend()
plt.show()



X_train shape: (1378, 280, 20)
y_train shape: (1378, 20)
X_test shape: (310, 280, 20)
y_test shape: (310, 20)

Fold 1
Epoch 1/100
30/30 - 12s - 407ms/step - loss: 0.1869 - mae: 0.0920 - val_loss: 0.0625 - val_mae: 0.0476 - learning_rate: 0.0010
Epoch 2/100
30/30 - 8s - 252ms/step - loss: 0.0247 - mae: 0.0285 - val_loss: 0.0524 - val_mae: 0.0378 - learning_rate: 0.0010
Epoch 3/100
30/30 - 8s - 279ms/step - loss: 0.0145 - mae: 0.0192 - val_loss: 0.0506 - val_mae: 0.0357 - learning_rate: 0.0010
Epoch 4/100
30/30 - 8s - 269ms/step - loss: 0.0122 - mae: 0.0165 - val_loss: 0.0507 - val_mae: 0.0353 - learning_rate: 0.0010
Epoch 5/100
30/30 - 10s - 339ms/step - loss: 0.0118 - mae: 0.0159 - val_loss: 0.0514 - val_mae: 0.0354 - learning_rate: 0.0010
Epoch 6/100
30/30 - 10s - 321ms/step - loss: 0.0105 - mae: 0.0142 - val_loss: 0.0440 - val_mae: 0.0336 - learning_rate: 0.0010
Epoch 7/100
30/30 - 9s - 308ms/step - loss: 0.0100 - mae: 0.0137 - val_loss: 0.0540 - val_mae: 0.0342 - learning_rate: 0.00

Epoch 13/100

Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
87/87 - 18s - 206ms/step - loss: 0.0202 - mae: 0.0225 - val_loss: 0.0332 - val_mae: 0.0270 - learning_rate: 0.0010
Epoch 14/100


In [ ]:
from scipy.stats import pearsonr
pearson_corr, _ = pearsonr(y_test_inv.flatten(), y_pred_inv.flatten())
print(f"Pearson Correlation (R): {pearson_corr:.4f}")

In [ ]:
from scipy.stats import spearmanr

spearman_corr, _ = spearmanr(y_test_inv.flatten(),y_pred_inv.flatten())
print(f"Spearman correlation: {spearman_corr:.4f}")

In [ ]:
direction_truth = np.sign(np.diff(y_test_inv.flatten()))
direction_pred = np.sign(np.diff(y_pred_inv.flatten()))
directional_accuracy = np.mean(direction_truth == direction_pred)
print(f"Directional accuracy: {directional_accuracy:.4f}")

In [ ]:
from sklearn.metrics import r2_score
y_pred = model.predict(X_val)
print("R2 Score:", r2_score(y_val, y_pred))

In [ ]:
import numpy as np
print("Predicted std per sample:", np.std(y_pred, axis=1).mean())

In [ ]:

for i in range(0,20):
    plt.figure(figsize=(10, 4))
    plt.plot(y_test_inv[i], label='real')
    plt.plot(y_pred_inv[i], label='perdict')
    plt.legend()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def predict_and_plot(df, model, scaler, features, target_date):
    day_steps = 30  
    input_days = 14

  
    start_date = target_date - pd.Timedelta(days=input_days)
    input_window = df.loc[(df.index >= start_date) & (df.index < target_date)]

    if len(input_window) != input_days * day_steps:
        raise ValueError({input_days*day_steps}, {len(input_window)} )

    input_scaled = scaler.transform(input_window[features])
    input_scaled = input_scaled.reshape(1, input_days * day_steps, len(features))

 
    y_pred_scaled = model.predict(input_scaled)

 
    y_pred_full = np.concatenate([
        y_pred_scaled.reshape(-1, 1),
        np.zeros((day_steps, len(features) - 1))
    ], axis=1)

    y_pred_inv = scaler.inverse_transform(y_pred_full)[:, 0]  


    forecast_index = pd.date_range(start=target_date, periods=day_steps, freq='20T')
    y_true = df.loc[forecast_index, 'Aggregate_mean'].values


    plt.figure(figsize=(12, 5))
    plt.plot(forecast_index, y_true, label='Actual', linewidth=2)
    plt.plot(forecast_index, y_pred_inv, label='Predicted', linestyle='--')
    plt.title(f"Prediction for {target_date.strftime('%Y-%m-%d')}")
    plt.xlabel("Time")
    plt.ylabel("Aggregate Load")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


predict_and_plot(ex4, model, scaler, features, pd.to_datetime('2014-09-11'))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta


start_date = '2014-08-13 00:00:00'
end_date = '2014-08-26 10:00:00'


input_data = ex4.loc[start_date:end_date, features]
print(f"Input data length: {len(input_data)}")
assert len(input_data) == 420


input_scaled = scaler.transform(input_data)
input_scaled = input_scaled.reshape((1, 420, len(features)))  # (batch_size=1, time_steps=420, features)


pred = model.predict(input_scaled)
pred_full = np.concatenate([pred.reshape(-1, 1)] + [np.zeros((pred.shape[1], len(features)-1))], axis=1)
pred_inv = scaler.inverse_transform(pred_full)[:, 0] 


pred_time_index = pd.date_range(start='2014-08-27 00:00:00', periods=30, freq='20min')


actual_data = ex4.loc[pred_time_index, 'Aggregate_mean']
assert len(actual_data) == len(pred_time_index)


def detect_wakeup_flexible(series, window_size=3, threshold_increase=90, allow_drop=1):
    values = series.values
    for i in range(len(values) - window_size):
        window = values[i:i + window_size]
        drops = sum(1 for x, y in zip(window, window[1:]) if y < x)
        total_diff = window[-1] - window[0]
        if total_diff > threshold_increase and drops <= allow_drop:
            return series.index[i]
    return None


actual_wakeup_times = []
for day in pred_time_index.normalize().unique():
    day_truth = ex4.loc[day:day + pd.Timedelta(days=1), 'Aggregate_mean']
    wake_time = detect_wakeup_flexible(day_truth)
    if wake_time and (pred_time_index[0] <= wake_time <= pred_time_index[-1]):
        actual_wakeup_times.append(wake_time)


predicted_wakeup_times = []
pred_series = pd.Series(pred_inv, index=pred_time_index)

for day in pred_time_index.normalize().unique():
    day_pred = pred_series[day:day + pd.Timedelta(days=1)]
    wake_time = detect_wakeup_flexible(day_pred)
    if wake_time:
        predicted_wakeup_times.append(wake_time)


print("30 minutes threshold")
time_diff_threshold = pd.Timedelta(minutes=30)

for actual, pred in zip(actual_wakeup_times, predicted_wakeup_times):
    if actual and pred:
        time_diff = abs(actual - pred)
        if time_diff > time_diff_threshold:
            print(f"Alarm: actual wakeup {actual.strftime('%Y-%m-%d %H:%M')}, predicted {pred.strftime('%Y-%m-%d %H:%M')}")


residual = actual_data.values - pred_inv
abs_residual = np.abs(residual)

mean_res = np.mean(abs_residual)
std_res = np.std(abs_residual)
z_scores = (abs_residual - mean_res) / std_res

z_threshold = 1.0
alert_mask = z_scores > z_threshold
alert_times = actual_data.index[alert_mask]


plt.figure(figsize=(12, 6))
plt.plot(pred_time_index, pred_inv, label='Predict', marker='o')
plt.plot(pred_time_index, actual_data.values, label='Truth', linestyle='--', marker='x', color='orange')

# 画真实起床时间线蓝色
for t in actual_wakeup_times:
    plt.axvline(t, color='blue', linestyle='--', alpha=1, label='Actual wakeup' if t == actual_wakeup_times[0] else "")

# 画预测起床时间线绿色
for t in predicted_wakeup_times:
    plt.axvline(t, color='green', linestyle='--', alpha=1, label='Predict wakeup' if t == predicted_wakeup_times[0] else "")

# 画残差异常的报警红色
for i, alert_time in enumerate(alert_times):
    plt.axvspan(alert_time - pd.Timedelta(minutes=10), alert_time + pd.Timedelta(minutes=10), 
                color='red', alpha=0.3, label='Alarm time' if i == 0 else "")

plt.xlabel("Time")
plt.ylabel("Aggregate_mean")
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



from sklearn.model_selection import TimeSeriesSplit


features = ['Aggregate_mean', 'Load_Derivative', 'Max_Load_Derivative_60min','Weekday', '60min_variance', 'MRoC_60min', 'hour', 'month']


train_data = [ex0, ex1, ex2, ex3]
test_data = [ex4]

step_per_day = 30
input_days = 14
forecast_horizon = 30


train_concat = pd.concat(train_data, axis=0)
scaler = MinMaxScaler()
scaler.fit(train_concat[features])


X_house_list, y_house_list = [], []


for house_data in train_data:
    X_list, y_list = [], []
    anchors = house_data.index[house_data.index.time == pd.to_datetime('00:00').time()].tolist()
    for i in range(len(anchors) - input_days - 1):
        input_start = anchors[i]
        input_end = anchors[i + input_days]
        target_start = input_end
        target_end = anchors[i + input_days + 1]

        input_window = house_data.loc[input_start:input_end - pd.Timedelta(minutes=20)]
        target_window = house_data.loc[target_start:target_end - pd.Timedelta(minutes=20)]

        if len(input_window) == step_per_day * input_days and len(target_window) == forecast_horizon:
            input_scaled = scaler.transform(input_window[features])
            target_scaled = scaler.transform(target_window[features])
            X_list.append(input_scaled)
            y_list.append(target_scaled[:, features.index('Aggregate_mean')])
    
    X_house_list.append(np.array(X_list))
    y_house_list.append(np.array(y_list))


X_train = np.concatenate(X_house_list, axis=0)
y_train = np.concatenate(y_house_list, axis=0)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

X_test_list, y_test_list = [], []

for house_data in test_data:
    anchors = house_data.index[house_data.index.time == pd.to_datetime('00:00').time()].tolist()
    for i in range(len(anchors) - input_days - 1):
        input_start = anchors[i]
        input_end = anchors[i + input_days]
        target_start = input_end
        target_end = anchors[i + input_days + 1]

        input_window = house_data.loc[input_start:input_end - pd.Timedelta(minutes=20)]
        target_window = house_data.loc[target_start:target_end - pd.Timedelta(minutes=20)]

        if len(input_window) == step_per_day * input_days and len(target_window) == forecast_horizon:
            input_scaled = scaler.transform(input_window[features])
            target_scaled = scaler.transform(target_window[features])
            X_test_list.append(input_scaled)
            y_test_list.append(target_scaled[:, features.index('Aggregate_mean')])


X_test = np.array(X_test_list)
y_test = np.array(y_test_list)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


house_data = list(zip(X_house_list, y_house_list))
house_data_shuffled = shuffle(house_data, random_state=42)

X_train = np.concatenate([x for x, _ in house_data_shuffled], axis=0)
y_train = np.concatenate([y for _, y in house_data_shuffled], axis=0)


@tf.function
def soft_dtw_distance(y_true, y_pred, gamma=1.0):
    if len(y_true.shape) == 3:
        y_true = tf.squeeze(y_true, axis=-1)
    if len(y_pred.shape) == 3:
        y_pred = tf.squeeze(y_pred, axis=-1)
    
    batch_size = tf.shape(y_true)[0]
    seq_len = tf.shape(y_true)[1]

    def pairwise_dist(a, b):
        a = tf.expand_dims(a, 2)
        b = tf.expand_dims(b, 1)
        return tf.square(a - b)

    D = pairwise_dist(y_true, y_pred)

    R = tf.ones_like(D) * 1e8
    R = tf.tensor_scatter_nd_update(
        R,
        tf.concat([
            tf.expand_dims(tf.range(batch_size), axis=1),
            tf.zeros((batch_size, 1), dtype=tf.int32),
            tf.zeros((batch_size, 1), dtype=tf.int32)
        ], axis=1),
        tf.zeros((batch_size,))
    )

    for i in tf.range(1, seq_len):
        for j in tf.range(1, seq_len):
            r0 = R[:, i - 1, j - 1]
            r1 = R[:, i - 1, j]
            r2 = R[:, i, j - 1]
            r_min = tf.minimum(tf.minimum(r0, r1), r2)
            update = D[:, i, j] + r_min
            R = tf.tensor_scatter_nd_update(
                R,
                tf.concat([
                    tf.expand_dims(tf.range(batch_size), axis=1),
                    tf.fill((batch_size, 1), i),
                    tf.fill((batch_size, 1), j)
                ], axis=1),
                update
            )

    return tf.reduce_mean(R[:, -1, -1])


def combined_loss(alpha=0.5, beta=0.5):
    mae_fn = tf.keras.losses.MeanAbsoluteError()
    def loss_fn(y_true, y_pred):
        mae = mae_fn(y_true, y_pred)
        dtw = soft_dtw_distance(y_true, y_pred)
        return alpha * mae + beta * dtw
    return loss_fn

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    print(f"\nFold {fold+1}")
    
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

   
    input_seq = Input(shape=(X_train.shape[1], X_train.shape[2]))
    conv1 = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(input_seq)
    conv2 = Conv1D(filters=128, kernel_size=3, activation='relu', padding='same')(conv1)
    pool = MaxPooling1D(pool_size=2)(conv2)
    norm = BatchNormalization()(pool)
    lstm1 = LSTM(512, return_sequences=True, dropout=0.3)(norm)
    lstm2 = LSTM(256, return_sequences=False, dropout=0.3)(lstm1)
    output = Dense(forecast_horizon)(lstm2)
    model = Model(inputs=input_seq, outputs=output)
    model.compile(optimizer='adam', loss=combined_loss(), metrics=['mae'])

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
        ModelCheckpoint(f'best_model_fold{fold+1}.keras', monitor='val_loss', save_best_only=True)
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=16,
        callbacks=callbacks,
        verbose=2
    )


y_pred = model.predict(X_test)


y_pred_inv = scaler.inverse_transform(
    np.concatenate([y_pred.reshape(-1, 1)] + [np.zeros((y_pred.size, len(features) - 1))], axis=1)
)[:, 0].reshape(y_pred.shape)

y_test_inv = scaler.inverse_transform(
    np.concatenate([y_test.reshape(-1, 1)] + [np.zeros((y_test.size, len(features) - 1))], axis=1)
)[:, 0].reshape(y_test.shape)


mse = mean_squared_error(y_test_inv.flatten(), y_pred_inv.flatten())
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_inv.flatten(), y_pred_inv.flatten())

print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MSE: {mse:.6f}")


plt.figure(figsize=(10, 4))
plt.plot(y_test_inv[0], label='truth')
plt.plot(y_pred_inv[0], label='predict')
plt.legend()
plt.show()
